# SNValue Merge + Phenotype Annotation Pipeline

**Purpose.** Reproduce the `3_16_26_manual_annotation.csv` output format starting from the two new raw files:
- `SNValueQuery_398645_20250528_20260528 (1).csv`
- `SNValueQuery_401430_20250528_20260528 (1).csv`

by (1) row-stacking the two files and (2) applying the logics from `phenotype.ipynb`.

**User-confirmed decisions (do not change without asking):**
1. **Merge = vertical stack** (row-concatenate, align on the shared 8 columns).
2. **Derivations:** `breeding_date = date_collected − DSLB days`; `combined_cow = CowID.0ClientID.0` (e.g. `1 + 392159 → 1.0392159.0`). Verified against `unique_combination_cleaned.csv` (e.g. `3/4/2022 − 35 = 1/28/2022`).
3. **Breeding-event clustering = phenotype Cell 7:** group by `(combined_cow, year, month)`, `day.diff() >= 2` starts a new event. `adj_final_breeding = min(breeding_date)` per cluster; `breed_event_id = cow_year_month` (single cluster) or `cow_year_month_cluster` (multi-cluster) — faithful to Cell 7.
4. **`final_resolved` = phenotype Cell 54** (latest fixed `resolve_cow_history` with chronological breed events, gap `<300` rebred overrides, single-sample rebred fixes, last-event handling, including the `Pregant_with_rc` spelling).
5. **`first_sample_rc`, `rc_resolved_open` left blank/NaN**; missing `Date Received/Sent/Approved`, `ProjectNo` left as NaN (new raw files do not contain them).

**Pipeline output:** `merged_annotation_output.csv` with the exact 23-column order of `3_16_26_manual_annotation.csv`.

**Performance note:** the literal Cell-7 Python loop over ~11k `(cow, year, month)` groups takes >60 s and the per-cow `groupby.apply` resolve takes ~100 s. This notebook therefore implements **vectorized equivalents with identical semantics** (documented inline) so the full run finishes in ~100 s instead of timing out. The original loop code is quoted in comments for traceability.


## 0. Imports & paths

In [ ]:
import time
import numpy as np
import pandas as pd

T0 = time.time()
def log(msg):
    print(f"[{time.time()-T0:.1f}s] {msg}", flush=True)

F1 = '/home/rajesh/work/ornella/SNValueQuery_398645_20250528_20260528 (1).csv'
F2 = '/home/rajesh/work/ornella/SNValueQuery_401430_20250528_20260528 (1).csv'
MANUAL_REF = '/home/rajesh/work/ornella/3_16_26_manual_annotation.csv'
OUT = '/home/rajesh/work/ornella/merged_annotation_output.csv'


## Step 1 — Merge the two SNValue files (row-stack)

Both raw files share the same 8 columns (`Date Collected, CowID, ClientID, Test, Result, S-N Value, DIM, DSLB`) for different `ClientID`s (398645 vs 401430). We concatenate rows, then drop the fully-empty trailer rows (the exports end with blank lines that parse as all-NaN).


In [ ]:
df1 = pd.read_csv(F1, dtype=str, keep_default_na=True)
df2 = pd.read_csv(F2, dtype=str, keep_default_na=True)
log(f"raw rows: file1={len(df1)} file2={len(df2)}")
df = pd.concat([df1, df2], ignore_index=True)
df = df.dropna(how='all').reset_index(drop=True)
log(f"stacked after dropping all-NaN trailer rows: {len(df)}")
df.head(3)


## Step 2 — Clean keys (`Date Collected`, `CowID`, `ClientID`, `DIM`, `DSLB`, `S-N Value`)

- `Date Collected` has a leading vertical-tab (`\x0b`, visible in raw bytes). Strip all control chars (`[\x00-\x1f\x7f]`) and whitespace.
- Drop rows missing `Date Collected` / `CowID` / `ClientID` (needed for cow history + `combined_cow`).
- `DIM` / `DSLB` → numeric; drop rows with missing `DSLB` (cannot derive `breeding_date`).
- `S-N Value` kept as **string** on purpose: many `Pregnant` rows in the 398645 file carry the literal value `POS` instead of a number; old data was all-numeric but we must not coerce `POS` to NaN.
- `Result` in new data includes `Not Tested` (52 rows) which did not exist in old data — kept as-is; Cell-54 has no rule for it so those events resolve to `NaN` (see QC step).


In [ ]:
# control-char strip (\x0b etc.)
df['Date Collected'] = (
    df['Date Collected'].astype(str)
      .str.replace(r'[\x00-\x1f\x7f]+', '', regex=True)
      .str.strip()
)
df['Date Collected'] = df['Date Collected'].replace({'': np.nan, 'nan': np.nan, 'NaT': np.nan, 'None': np.nan})
for c in ['CowID', 'ClientID']:
    df[c] = df[c].astype(str).str.strip().replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
df = df[df['Date Collected'].notna() & df['CowID'].notna() & df['ClientID'].notna()].copy()

df['DIM'] = pd.to_numeric(df['DIM'], errors='coerce')
df['DSLB'] = pd.to_numeric(df['DSLB'], errors='coerce')
log(f"DSLB NaN={int(df['DSLB'].isna().sum())}, DIM NaN={int(df['DIM'].isna().sum())}")
df = df[df['DSLB'].notna()].copy()
df['S-N Value'] = df['S-N Value'].astype(str).str.strip().replace({'nan': np.nan, '': np.nan, 'None': np.nan})
log(f"after key cleaning: {len(df)} rows")
print(df['Result'].value_counts(dropna=False))


## Step 3 — Derive `date_collected`, `breeding_date`, `combined_cow`, `year/month/day`

Mirrors phenotype Cell 6 plus the pre-step that built `unique_combination_cleaned.csv`:
- `date_collected_dt = pd.to_datetime(Date Collected)`; `date_collected` string kept as `M/D/YYYY` (no leading zeros) to match the manual file.
- `breeding_date = date_collected_dt − DSLB days` (timedelta). Check: `3/4/2022 − 35d = 1/28/2022`.
- `combined_cow = f"{CowID_int}.0{ClientID_int}.0"` → `1 + 392159 = 1.0392159.0`; `1000 + 401430 = 1000.0401430.0`. Integer-cast via `int(float(x))` so `"1000.0"` inputs also work.
- `year/month/day/month_day` from `breeding_date` (Cell 6).


In [ ]:
df['date_collected_dt'] = pd.to_datetime(df['Date Collected'], format='mixed', errors='coerce')
log(f"date parse failures: {int(df['date_collected_dt'].isna().sum())}")
df = df[df['date_collected_dt'].notna()].copy()
df['breeding_date'] = df['date_collected_dt'] - pd.to_timedelta(df['DSLB'].astype(int), unit='D')

def build_combined(cow, cli):
    try: cow_s = str(int(float(str(cow).strip())))
    except Exception: cow_s = str(cow).strip()
    try: cli_s = str(int(float(str(cli).strip())))
    except Exception: cli_s = str(cli).strip()
    return f"{cow_s}.0{cli_s}.0"
df['combined_cow'] = [build_combined(a, b) for a, b in zip(df['CowID'].values, df['ClientID'].values)]

df['date_collected'] = df['date_collected_dt'].dt.strftime('%-m/%-d/%Y')
df['Date Collected'] = df['date_collected']  # canonical cleaned form
# Cell 6
df['year'] = df['breeding_date'].dt.year
df['month'] = df['breeding_date'].dt.month
df['day'] = df['breeding_date'].dt.day
df['month_day'] = df['breeding_date'].dt.strftime('%m-%d')
log(f"cows={df['combined_cow'].nunique()}, breeding {df['breeding_date'].min().date()}..{df['breeding_date'].max().date()}")
df[['Date Collected','CowID','ClientID','combined_cow','date_collected','breeding_date','DSLB','year','month','day']].head(3)


## Step 4 — Breeding-event clustering (phenotype Cell 7, vectorized)

**Original Cell-7 logic (quoted for traceability):** group by `(combined_cow, year, month)`; within each group `date_diff = day.diff()`; `is_new_event = (date_diff >= 2).fillna(True)`; `cluster = is_new_event.cumsum()`; `adj_final_breeding = min(breeding_date)` per cluster; `breed_event_id = cow_year_month_cluster` if >1 cluster else `cow_year_month` (no suffix).

**Two deliberate, documented choices:**
1. *Sort* each `(cow, year, month)` group by `breeding_date` before `.diff()`. The notebook relied on input order; sorting makes the result deterministic and matches the intent (re-breeds 1 day apart stay together, ≥2 days split).
2. *Vectorize* the loop (`sort → groupby.diff → cumsum → groupby.transform('min')`) — identical output, ~2 s instead of >60 s for ~11k groups. `nclus = nunique(cluster)` per group reproduces the with/without-suffix branch.

Known format difference vs the manual file: faithful Cell 7 leaves ~95% of IDs **without** a cluster suffix (`cow_year_month`), while `3_16_26_manual_annotation.csv` always has one (`cow_year_month_cluster`). That is expected from the chosen Cell-7 branch — see QC step.


In [ ]:
# Faithful-but-vectorized Cell 7. Compare with the loop quoted above.
df = df.sort_values(['combined_cow', 'year', 'month', 'breeding_date']).reset_index(drop=True)
g = df.groupby(['combined_cow', 'year', 'month'], sort=False)
df['date_diff'] = g['day'].diff()
df['check_new_event'] = (df['date_diff'] >= 2) | df['date_diff'].isna()
df['cluster'] = g['check_new_event'].cumsum()
df['adj_final_breeding'] = df.groupby(['combined_cow', 'year', 'month', 'cluster'], sort=False)['breeding_date'].transform('min')
df['nclus'] = df.groupby(['combined_cow', 'year', 'month'], sort=False)['cluster'].transform('nunique')
base = df['combined_cow'].astype(str) + '_' + df['year'].astype(str) + '_' + df['month'].astype(str)
df['breed_event_id'] = base + np.where(df['nclus'] > 1, '_' + df['cluster'].astype(str), '')
df = df.drop(columns=['nclus'])
log(f"cell7 done: rows={len(df)}, events={df['breed_event_id'].nunique()}")
print((df['breed_event_id'].str.count('_') + 1).value_counts().rename(index={3: '3-part (no suffix)', 4: '4-part (with suffix)'}))
df[df['combined_cow'] == '10010.0401430.0'][['breeding_date','adj_final_breeding','day','breed_event_id','Result','DSLB']].head(10)


## Step 5 — `ai_id` (phenotype Cell 13)

Per cow, rank distinct `adj_final_breeding` values chronologically `1..n`. Vectorized as `groupby('combined_cow')['adj_final_breeding'].rank(method='dense')` — identical to the notebook's `sorted(unique) → {date: i+1}` loop.


In [ ]:
df['adj_final_breeding'] = pd.to_datetime(df['adj_final_breeding'])
df['ai_id'] = df.groupby('combined_cow', sort=False)['adj_final_breeding'].rank(method='dense').astype(int)
log('ai_id done')
df[['combined_cow','adj_final_breeding','ai_id','breed_event_id']].head(8)


## Step 6 — `sample_id` (phenotype Cell 15)

Per `(combined_cow, ai_id)`, rank distinct `date_collected` values chronologically `1..n` after sorting by `date_collected`. Vectorized dense rank — identical to the notebook loop.


In [ ]:
df['sample_id'] = df.groupby(['combined_cow', 'ai_id'], sort=False)['date_collected_dt'].rank(method='dense').astype(int)
log('sample_id done')
df[['combined_cow','ai_id','date_collected','sample_id']].head(8)


## Step 7 — `sample_class` (phenotype Cell 17)

Per `breed_event_id`: `sample_id == 1 → first`; `== max → last`; else `intermediate`. Vectorized with `groupby.transform('max')`.


In [ ]:
mx = df.groupby('breed_event_id', sort=False)['sample_id'].transform('max')
df['sample_class'] = np.where(df['sample_id'] == 1, 'first', np.where(df['sample_id'] == mx, 'last', 'intermediate'))
log(str(df['sample_class'].value_counts().to_dict()))
df[['breed_event_id','sample_id','sample_class','Result']].head(10)


## Step 8 — `final_resolved` (phenotype Cell 54, latest fixed `resolve_cow_history`)

Applied verbatim, including the `Pregant_with_rc` spelling. Per cow (events in chronological order):

| n | condition | `final_resolved` |
|---|-----------|------------------|
| 1 | — | the single `Result` |
| >1, ends `Pregnant` | first `Pregnant`, has `Open` | `preg_open_preg` |
| >1, ends `Pregnant` | first `Pregnant`, has `Re-Check` | `Pregant_with_rc` |
| >1, ends `Pregnant` | first `Pregnant`, else | `Pregnant` |
| >1, ends `Pregnant` | first `Re-Check` | `rc_preg` |
| >1, ends `Pregnant` | first `Open` | `open_to_preg` |
| >1, ends `Open` | has `Pregnant` | `loss` |
| >1, ends `Open` | first `Re-Check` | `rc_open` |
| >1, ends `Open` | else | `Open` |
| >1, ends `Re-Check` | has `Pregnant` | `preg_to_rc_unresolved` |
| >1, ends `Re-Check` | has `Open` | `open_to_rc_unresolved` |
| >1, ends `Re-Check` | else | `rc_unresolved` |

**Gap overrides** (`curr` → next `adj_final_breeding`): single-event cows ending `Re-Check` → `rc_unresolved`, ending `Open` with any `Pregnant` → `loss_2_open`; multi-event non-last events with gap `<300` and last ≠ `Open` map all-`Pregnant` → `preg_loss_rebred`, `Pregnant`+`Re-Check` → `rc_preg_loss_rebred`, all-`Re-Check` → `rc_rebred` (plus single-sample rebred fixes); gap `>=300` with last `Re-Check` → `rc_unresolved_>_1_bred`; last event ending `Re-Check` → `rc_unresolved_>_1_bred`, ending `Open` with `Pregnant` → `loss_2_open`.

`Not Tested` has no rule and stays `NaN` (see QC). Implementation below loops per cow like the notebook but avoids `groupby.apply` overhead (uses `groupby.indices` + vectorized masks).


In [ ]:
df['final_resolved'] = None
df = df.sort_values(['combined_cow', 'adj_final_breeding', 'sample_id']).reset_index(drop=True)

def resolve_cow_history_inline(cow, idx):
    """Cell-54 logic for one cow; writes directly into df via .loc."""
    sub = df.loc[idx].sort_values(['adj_final_breeding', 'sample_id'])
    bed = sub.groupby('breed_event_id', sort=False)['adj_final_breeding'].min().sort_values()
    events = bed.index.tolist()
    emin = bed.to_dict()
    n_events = len(events)
    for i, eid in enumerate(events):
        m = df.index.isin(idx) & (df['breed_event_id'].values == eid)
        ev_rows = df.loc[m].sort_values('sample_id')
        samples = ev_rows['Result'].tolist()
        n = len(samples)
        if n == 0:
            continue
        first, last = samples[0], samples[-1]
        resolved = None
        if n == 1:
            resolved = first
        else:
            if last == 'Pregnant':
                if first == 'Pregnant':
                    if 'Open' in samples: resolved = 'preg_open_preg'
                    elif 'Re-Check' in samples: resolved = 'Pregant_with_rc'
                    else: resolved = 'Pregnant'
                elif first == 'Re-Check': resolved = 'rc_preg'
                elif first == 'Open': resolved = 'open_to_preg'
            elif last == 'Open':
                if 'Pregnant' in samples: resolved = 'loss'
                elif first == 'Re-Check': resolved = 'rc_open'
                else: resolved = 'Open'
            elif last == 'Re-Check':
                if 'Pregnant' in samples: resolved = 'preg_to_rc_unresolved'
                elif 'Open' in samples: resolved = 'open_to_rc_unresolved'
                else: resolved = 'rc_unresolved'
        if n_events == 1:
            if last == 'Re-Check': resolved = 'rc_unresolved'
            elif last == 'Open' and 'Pregnant' in samples: resolved = 'loss_2_open'
        else:
            if i < n_events - 1:
                gap = (emin[events[i+1]] - emin[eid]).days
                if gap < 300:
                    if last != 'Open':
                        if all(r == 'Pregnant' for r in samples): resolved = 'preg_loss_rebred'
                        elif 'Pregnant' in samples and 'Re-Check' in samples: resolved = 'rc_preg_loss_rebred'
                        elif all(r == 'Re-Check' for r in samples): resolved = 'rc_rebred'
                        elif n == 1 and last == 'Re-Check': resolved = 'rc_rebred'
                        elif n == 1 and last == 'Pregnant': resolved = 'preg_loss_rebred'
                elif gap >= 300:
                    if last == 'Re-Check': resolved = 'rc_unresolved_>_1_bred'
            else:
                if last == 'Re-Check': resolved = 'rc_unresolved_>_1_bred'
                elif last == 'Open' and 'Pregnant' in samples: resolved = 'loss_2_open'
        df.loc[m, 'final_resolved'] = resolved

cow_groups = df.groupby('combined_cow', sort=False).indices
log(f"resolving {len(cow_groups)} cows ...")
for k, (cow, idx) in enumerate(cow_groups.items(), 1):
    resolve_cow_history_inline(cow, idx)
    if k % 2000 == 0:
        log(f"resolved {k}/{len(cow_groups)} cows")
log(f"resolve done; unresolved(NaN)={int(df['final_resolved'].isna().sum())}")
print(df['final_resolved'].value_counts(dropna=False).head(30))


## Step 9 — Format output like `3_16_26_manual_annotation.csv`

Exact 23-column order: `Date Collected, Date Received, CowID, Date Sent, Date Approved, ClientID, ProjectNo, Test, S-N Value, DIM, combined_cow, date_collected, year, adj_final_breeding, DSLB, breed_event_id, ai_id, sample_id, sample_class, Result, final_resolved, first_sample_rc, rc_resolved_open`.
- `Date Received/Sent/Approved`, `ProjectNo`, `first_sample_rc`, `rc_resolved_open` = NaN (absent from new raw files; last two per user choice).
- `adj_final_breeding` formatted `M/D/YYYY` (no leading zeros) like the manual file.
- Sorted by `combined_cow, adj_final_breeding, date_collected` like the manual file.


In [ ]:
for c in ['Date Received', 'Date Sent', 'Date Approved', 'ProjectNo']:
    df[c] = np.nan
df['first_sample_rc'] = np.nan
df['rc_resolved_open'] = np.nan
df['adj_str'] = pd.to_datetime(df['adj_final_breeding']).dt.strftime('%-m/%-d/%Y')

def clean_id(x):
    try:
        f = float(str(x))
        return str(int(f)) if f.is_integer() else str(x)
    except Exception:
        return str(x)
df['CowID'] = df['CowID'].apply(clean_id)
df['ClientID'] = df['ClientID'].apply(clean_id)

out = pd.DataFrame({
    'Date Collected': df['Date Collected'],
    'Date Received': df['Date Received'],
    'CowID': df['CowID'],
    'Date Sent': df['Date Sent'],
    'Date Approved': df['Date Approved'],
    'ClientID': df['ClientID'],
    'ProjectNo': df['ProjectNo'],
    'Test': df['Test'],
    'S-N Value': df['S-N Value'],
    'DIM': df['DIM'],
    'combined_cow': df['combined_cow'],
    'date_collected': df['date_collected'],
    'year': df['year'],
    'adj_final_breeding': df['adj_str'],
    'DSLB': df['DSLB'],
    'breed_event_id': df['breed_event_id'],
    'ai_id': df['ai_id'],
    'sample_id': df['sample_id'],
    'sample_class': df['sample_class'],
    'Result': df['Result'],
    'final_resolved': df['final_resolved'],
    'first_sample_rc': df['first_sample_rc'],
    'rc_resolved_open': df['rc_resolved_open'],
})
out['_b'] = pd.to_datetime(df['adj_final_breeding'].values)
out['_c'] = df['date_collected_dt'].values
out = out.sort_values(['combined_cow', '_b', '_c']).drop(columns=['_b', '_c']).reset_index(drop=True)
out.to_csv(OUT, index=False)
log(f"wrote {OUT} rows={len(out)}")
out.head(10)


## Step 10 — Verification / QC

1. Column order matches the manual file.
2. Overlap check: cow `10010.0401430.0` (present in both eras, breeding `5/21/2025`) should show the same 5-sample `Pregnant → Re-Check ×3 → Pregnant` sequence resolving to `Pregant_with_rc`.
3. Unresolved (`NaN`) rows should only be `Not Tested`-mixed sequences, which Cell 54 does not cover.
4. `breed_event_id` parts check documents the faithful-Cell-7 suffix behavior.


In [ ]:
man_cols = list(pd.read_csv(MANUAL_REF, nrows=2).columns)
print('columns match manual:', list(out.columns) == man_cols)
print('out shape:', out.shape)
print(out['final_resolved'].value_counts(dropna=False))
print()
print('--- overlap cow 10010.0401430.0 ---')
print(out[out['combined_cow'] == '10010.0401430.0'][['Date Collected','DSLB','adj_final_breeding','breed_event_id','ai_id','sample_id','sample_class','Result','final_resolved']].to_string(index=False))
print()
print('--- unresolved patterns (expect only Not Tested mixes) ---')
un = out[out['final_resolved'].isna()]
print('unresolved rows:', len(un))
print(un.groupby('breed_event_id')['Result'].apply(list).apply(tuple).value_counts().head(10))
print()
print('--- breed_event_id suffix check (faithful Cell 7) ---')
print((out['breed_event_id'].str.count('_') + 1).value_counts().rename(index={3: '3-part (no suffix)', 4: '4-part (with suffix)'}))
print(f"total runtime: {time.time()-T0:.1f}s")
